# 16

In [1]:
import pandas as pd
from pathlib import Path
import os

current_dir = Path.cwd()
data_path = os.path.join(current_dir, "Industrial_and_Scientific.json")

print("current_dir:", current_dir)
print("data_path:", data_path)

current_dir: /Users/rebecca/Documents/CROUSE/Winter-2025/COMP262-NLP_RS/Project/W25_COMP262_002_TeamGamma
data_path: /Users/rebecca/Documents/CROUSE/Winter-2025/COMP262-NLP_RS/Project/W25_COMP262_002_TeamGamma/Industrial_and_Scientific.json


In [2]:
#load data
dataset = pd.read_json(data_path,orient='records',lines=True)

In [3]:
dataset.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1758333 entries, 0 to 1758332
Data columns (total 12 columns):
 #   Column          Non-Null Count    Dtype 
---  ------          --------------    ----- 
 0   overall         1758333 non-null  int64 
 1   verified        1758333 non-null  bool  
 2   reviewTime      1758333 non-null  object
 3   reviewerID      1758333 non-null  object
 4   asin            1758333 non-null  object
 5   reviewerName    1758223 non-null  object
 6   reviewText      1757349 non-null  object
 7   summary         1757930 non-null  object
 8   unixReviewTime  1758333 non-null  int64 
 9   vote            206308 non-null   object
 10  style           691514 non-null   object
 11  image           32710 non-null    object
dtypes: bool(1), int64(2), object(9)
memory usage: 149.2+ MB


In [4]:
#drop null column of reviewText, summary
dataset = dataset.dropna(subset=['reviewText'])
dataset = dataset.dropna(subset=['summary'])

In [5]:
dataset.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 1757002 entries, 0 to 1758332
Data columns (total 12 columns):
 #   Column          Non-Null Count    Dtype 
---  ------          --------------    ----- 
 0   overall         1757002 non-null  int64 
 1   verified        1757002 non-null  bool  
 2   reviewTime      1757002 non-null  object
 3   reviewerID      1757002 non-null  object
 4   asin            1757002 non-null  object
 5   reviewerName    1756892 non-null  object
 6   reviewText      1757002 non-null  object
 7   summary         1757002 non-null  object
 8   unixReviewTime  1757002 non-null  int64 
 9   vote            206218 non-null   object
 10  style           690977 non-null   object
 11  image           32455 non-null    object
dtypes: bool(1), int64(2), object(9)
memory usage: 162.5+ MB


In [6]:
dataset.head(5)

,overall,verified,reviewTime,reviewerID,asin,reviewerName,reviewText,summary,unixReviewTime,vote,style,image
0,5,True,"01 23, 2013",A3FANY5GOT5X0W,0176496920,Kelly Keyser,"Arrived on time, in mint condition, great! I ...",Just as described!,1358899200,NaN,NaN,NaN
1,5,True,"11 5, 2012",AT6HRPPYOPHMB,0176496920,Michael C,This device was hard to find for my daughter's...,Great device,1352073600,NaN,NaN,NaN
2,4,True,"10 17, 2012",A4IX7B38LIN1E,0176496920,BH,Just a clicker nothing special. Was hoping it ...,Pretty Good,1350432000,NaN,NaN,NaN
3,5,True,"03 29, 2017",A12Q4LR8N17AOZ,0176496920,Waterfall3500,Great response card. Slow shipping but it work...,Thank you for the great product. Works. A++ Us...,1490745600,NaN,NaN,NaN
4,1,True,"03 21, 2017",A1GJXZZPOZ3OD9,0176496920,Amazon Customer,It only lasted for 3 days before it stopped wo...,One Star,1490054400,NaN,NaN,NaN


In [7]:
# Count words in each reviewText
dataset['word_count'] = dataset['reviewText'].apply(lambda x: len(str(x).split()))

long_reviews = dataset[dataset['word_count'] > 100].copy()

In [8]:
import pandas as pd
import difflib
import re

def clean_text(text):
    return re.sub(r"\s+", " ", text.strip().lower())

def is_high_quality_summary(text, summary):
    text = clean_text(text)
    summary = clean_text(summary)
    
    # compare length
    text_len = len(text.split())
    summary_len = len(summary.split())
    compression_ratio = summary_len / max(text_len, 1)
    
    # conparessed
    is_compressed = compression_ratio < 0.35

    # match sequence words 
    matcher = difflib.SequenceMatcher(None, text, summary)
    long_matches = [block for block in matcher.get_matching_blocks() if block.size >= 30]  # 30 letters

    # match ratio
    match_ratio = sum([block.size for block in long_matches]) / max(len(summary), 1)
    is_not_copy = match_ratio < 0.5

    return is_compressed and is_not_copy

# good summary
def evaluate_summaries(df, text_col='reviewText', summary_col='summary'):
    df = df.copy()
    df["is_high_quality"] = df.apply(
        lambda row: is_high_quality_summary(row[text_col], row[summary_col]), axis=1
    )
    return df

In [9]:
df_filtered = evaluate_summaries(long_reviews)

high_quality = df_filtered[df_filtered['is_high_quality']]
low_quality = df_filtered[~df_filtered['is_high_quality']]

print(f"{len(high_quality)} / {len(long_reviews)}")

105941 / 120858


In [10]:
selected_reviews = high_quality.sample(10, random_state=42)

In [11]:
selected_reviews

,overall,verified,reviewTime,reviewerID,asin,reviewerName,reviewText,summary,unixReviewTime,vote,style,image,word_count,is_high_quality
734544,4,True,"01 15, 2014",AEV23XYLMVFZL,B00D7IEAY4,Marcelo Silva,I operate a commercial janitorial services com...,Great value,1389744000,94,NaN,NaN,144,True
789424,5,True,"01 6, 2016",A9AB5APVHTKX8,B00EUKHACW,Canna98,"Having used some of the so called ""Best"" vacuu...",What a Machine!,1452038400,6,{'Size:': ' HV302'},NaN,319,True
991859,5,True,"03 2, 2017",A16ZDXB314DTM6,B00T3ZRI44,Kory,worked for a week and then started making loud...,works only on Smart TV menus,1488412800,NaN,{'Color:': ' Black'},NaN,101,True
245121,5,True,"03 29, 2014",A2805TH6JI69SN,B000W22Z5Y,all the things,This is a great product for the price. I am a ...,Excellent product and company,1396051200,NaN,NaN,NaN,132,True
419395,3,True,"04 2, 2017",ASMTQGULYOY9O,B004AGJ2IG,SonSon,Not bad. I was looking for something to unclog...,Almost a really good product,1491091200,2,{'Package Quantity:': ' 1'},NaN,146,True
14420,3,False,"06 7, 2011",A3KS73J4FNRSJ2,B00005IBW1,Customer,This is a good bet for a generic drugstore whi...,Used this for years with good results,1307404800,4,"{'Size:': ' Kit', 'Style Name:': ' 2-Hour Whit...",NaN,226,True
1199185,5,True,"07 6, 2017",A243TDF8WBG23F,B01FZXLN5C,markmagoo,"If you work in live production, STOP USING DUC...",THE TAPE for live productions (and life),1499299200,NaN,"{'Size:': ' 4 Inch X 30 Yards', 'Color:': ' Bl...",NaN,102,True
1719875,1,True,"06 10, 2016",ABE3ALBA75IUB,B01BLYLUSY,LLShopper,"I've used other brands silica gel before this,...",Doesn't work at all.,1465516800,5,{'Size:': ' 5 lbs'},NaN,148,True
809720,5,True,"03 27, 2015",ADJVGATXSADYT,B00FX36W7O,fishki,I mean that quite literally. Just arrived toda...,This thing sucks.....,1427414400,11,{'Style:': ' Vacuum'},NaN,188,True
798439,5,True,"06 12, 2014",ATHG41G849GIF,B00F9X9WYI,ontopahill,"This thing is an awesome toy and very, very us...","VERY fun toy, but beware of accuracy",1402531200,NaN,{'Size:': ' GXMT38'},NaN,222,True


In [12]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"

In [13]:
# import tensorflow as tf
# tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

In [14]:
from transformers.pipelines import SUPPORTED_TASKS

print(SUPPORTED_TASKS.keys())

dict_keys(['audio-classification', 'automatic-speech-recognition', 'text-to-audio', 'feature-extraction', 'text-classification', 'token-classification', 'question-answering', 'table-question-answering', 'visual-question-answering', 'document-question-answering', 'fill-mask', 'summarization', 'translation', 'text2text-generation', 'text-generation', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-audio-classification', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-to-text', 'image-text-to-text', 'object-detection', 'zero-shot-object-detection', 'depth-estimation', 'video-classification', 'mask-generation', 'image-to-image'])


In [15]:
single_text = selected_reviews.iloc[1]['reviewText']
single_text

'Having used some of the so called "Best" vacuum cleaners ever made etc., I was and am still blown away by this machine!  I had my own cleaning business for 25 plus years and know more than a thing or two about what makes a great vacuum and what is just hype!!  I have seen people buy worthless piles of crap just because it was expensive...therefore it must be good right? NOPE!\nI have been searching for something that was not too expensive, smaller, easy to use and versatile, as I have all hardwood floors and one large area rug with low pile.  Dragging in a canister vac every couple of days really gets old and my OCedar microfiber push broom doesn\'t do the one carpet I have!  I could not bring myself to plop down $300.00 for a Dyson either!  I bought this on Cyber Monday for less than $112.00!\nThere are so many different attachments that come with it, I still have not used them all!  I was astounded at what it pulled out of my area rug, well disgusted is more the word!  The first use

In [16]:
from transformers import pipeline

In [17]:
summarizer_gl = pipeline("summarization", model="google-t5/t5-large", framework="pt")

# Apply summarizer to each review
selected_reviews['summary_model'] = selected_reviews['reviewText'].apply(
    lambda x: summarizer_gl(x, max_length=50, min_length=25, do_sample=False)[0]['summary_text']
)

Device set to use mps:0


In [18]:
print("Review1:",selected_reviews.iloc[0]['reviewText'])
print("\nSummary 1:", selected_reviews.iloc[0]['summary_model'])

Review1: I operate a commercial janitorial services company, and for the cost, these are the best backpack vacuums on the market. But it is not perfect.
Cons:
Some of the rivets may break, some sooner than expected (we now have 8 of these, so we bought a cheap rivet tool), and the electrical cables that goes into the vacuum will eventually get loose (they twist off by themselves, if you catch it early, no biggie), and I wish the top would latch on a bit tighter.
Replacement parts are expensive/hard to find
Pros:
Light, powerful and comfortable (this can not be overstated, the back harness is great)
Not a single issue in relation to the motor.
Quiet (in relation).

Are there better vacuums? Yes, but in the $350.00 to $425.00 range. As far as we are concerned, this is far and away the best value.

Summary 1: for the cost, these are the best backpack vacuums on the market, but it is not perfect . some of the rivets may break, some sooner than expected . replacement parts are expensive/har

In [19]:
print("Review2:",selected_reviews.iloc[1]['reviewText'])
print("\nSummary 2:", selected_reviews.iloc[1]['summary_model'])

Review2: Having used some of the so called "Best" vacuum cleaners ever made etc., I was and am still blown away by this machine!  I had my own cleaning business for 25 plus years and know more than a thing or two about what makes a great vacuum and what is just hype!!  I have seen people buy worthless piles of crap just because it was expensive...therefore it must be good right? NOPE!
I have been searching for something that was not too expensive, smaller, easy to use and versatile, as I have all hardwood floors and one large area rug with low pile.  Dragging in a canister vac every couple of days really gets old and my OCedar microfiber push broom doesn't do the one carpet I have!  I could not bring myself to plop down $300.00 for a Dyson either!  I bought this on Cyber Monday for less than $112.00!
There are so many different attachments that come with it, I still have not used them all!  I was astounded at what it pulled out of my area rug, well disgusted is more the word!  The firs

## Other test

In [20]:
output_gl = summarizer_gl(single_text, max_length=50, min_length=25, do_sample=False)[0]['summary_text']

print(output_gl)

i was and am still blown away by this machine! i had my own cleaning business for 25 plus years and know more than a thing or two about what makes a great vacuum and what is just hype . i


In [21]:
summarizer_g = pipeline("summarization", model="google/pegasus-xsum", framework="pt")
outputs_g = summarizer_g(single_text, max_length=50, min_length=25, do_sample=False)[0]['summary_text']

print(outputs_g)

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use mps:0


I have been looking for a vacuum cleaner for a long time and this is the one that I have been looking for!


In [22]:
summarizer_df = pipeline("summarization", model ="google-t5/t5-base", framework="pt")
outputs_df = summarizer_df(single_text, max_length=50,min_length=25, clean_up_tokenization_spaces=True)
print(outputs_df[0]['summary_text'])

Device set to use mps:0


cnn.com's ireport boot camp challenges you to find the best vacuum for your home. the OCedar microfiber push broom vacuum pulls out fine dirt every day for 2 weeks.


In [23]:
prompt = (
    "Summarize this customer review in 2-3 sentences. Highlight the main advantages and disadvantages expressed.Catch the user’s overall sentiment (positive/negative/mixed).Avoid copying the original text directly."
    + single_text
)

summary_df_p = summarizer_df(prompt, max_length=50, min_length=25, do_sample=False)[0]['summary_text']
print(summary_df_p)

i was and am still blown away by this machine! i bought this on cyber monday for less than $112.00!


In [24]:
summarizer_fb = pipeline("summarization", model ="facebook/bart-large-cnn", framework="pt")
outputs_fb = summarizer_fb(single_text, max_length=50,min_length=25, clean_up_tokenization_spaces=True)
print(outputs_fb[0]['summary_text'])

Device set to use mps:0


I was and am still blown away by this machine!  I had my own cleaning business for 25 plus years and know more than a thing or two about what makes a great vacuum. The deep down fine dirt that it pulled out,


In [26]:
pd.set_option('display.max_colwidth', None)

In [27]:
selected_reviews[['reviewText','summary']]

,reviewText,summary
734544,"I operate a commercial janitorial services company, and for the cost, these are the best backpack vacuums on the market. But it is not perfect.\nCons:\nSome of the rivets may break, some sooner than expected (we now have 8 of these, so we bought a cheap rivet tool), and the electrical cables that goes into the vacuum will eventually get loose (they twist off by themselves, if you catch it early, no biggie), and I wish the top would latch on a bit tighter.\nReplacement parts are expensive/hard to find\nPros:\nLight, powerful and comfortable (this can not be overstated, the back harness is great)\nNot a single issue in relation to the motor.\nQuiet (in relation).\n\nAre there better vacuums? Yes, but in the $350.00 to $425.00 range. As far as we are concerned, this is far and away the best value.",Great value
789424,"Having used some of the so called ""Best"" vacuum cleaners ever made etc., I was and am still blown away by this machine! I had my own cleaning business for 25 plus years and know more than a thing or two about what makes a great vacuum and what is just hype!! I have seen people buy worthless piles of crap just because it was expensive...therefore it must be good right? NOPE!\nI have been searching for something that was not too expensive, smaller, easy to use and versatile, as I have all hardwood floors and one large area rug with low pile. Dragging in a canister vac every couple of days really gets old and my OCedar microfiber push broom doesn't do the one carpet I have! I could not bring myself to plop down $300.00 for a Dyson either! I bought this on Cyber Monday for less than $112.00!\nThere are so many different attachments that come with it, I still have not used them all! I was astounded at what it pulled out of my area rug, well disgusted is more the word! The first use was impressive, but every day after that was well ridiculous! The deep down fine dirt that it pulled out, every day for 2 weeks, blew me away! Again, I was a professional housekeeper with my own business for 25+ years, I sweep with my OCedar everyday sometimes twice. Granted, I no longer have small children and a big dog running thru the house everyday, but my family still has no clue what wipe your feet or take them shoes off means! So being able to switch from the power head to the bare floor attachment as easily as you can and with such a long cord....I love it! Wish I had bought two! Or even three! I know a couple of family members who could use a machine like this!",What a Machine!
991859,"worked for a week and then started making loud buzzing sound. works only on Smart TV menus. once video starts playing the buzzing comes back. basically any video play back causes a buzzing\n\nUpdate 3/9/2017\nso the issue was not on the device but tv configuration. I was contact right away after my review. They were pleseant and helpful. I changed the audio format on the tv and boom... Sound from my audio receiver. This device works as avertised. If it does not work for you, check you tv audio settings. I will definitely buy from this company again. Thanks again!!!",works only on Smart TV menus
245121,"This is a great product for the price. I am a nursing student and its much better than the stethoscope that came with my books/tuition fees. It seems very comparable to some of the other students who have the Littmans. Not only is their product great, R.A. Bock the company is fantastic. I ordered a stethoscope from Amazon and upon inspection, noticed there was a small gash in the diaphragm that came on the stethoscope. I sent them an email and they rushed me a brand new stethoscope bubble wrapped for extra protection with a return envelope inside the package to send the defective one back. I would most definitely recommend the product and company. They were very easy to work with and happy to work with me to fix the defect.",Excellent product and company
419395,"Not bad. I was looking for something to unclog the drain pipe go

In [28]:
review = selected_reviews.iloc[0]['reviewText']

# 17

In [29]:
import pandas as pd
import re

question_words = ["what", "why", "how", "can", "could", "would", "do", "does", "did", "is", "are", "was", "were", "should", "may"]

def is_question_review(text):
    if "?" in text:
        return True
    return any(re.match(rf"^\s*{q}\b", text.strip().lower()) for q in question_words)

high_quality["is_question_like"] = high_quality["reviewText"].apply(is_question_review)

question_review = high_quality[high_quality["is_question_like"]].sample(10, random_state=42)

/var/folders/60/83kzkx2x4hs5zc8lx2tyc8lw0000gn/T/ipykernel_37836/81743553.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  high_quality["is_question_like"] = high_quality["reviewText"].apply(is_question_review)


In [30]:
question_review["reviewText"]

694454                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [31]:
text_question = high_quality["reviewText"][270552]
text_question

"I buy a lot of stuffs from Amazon and this would be my very first review.  As a contractor, I have other higher end backpack vacuum.  I was hoping to buy this smaller version to use around the house for simple use on all my hardwood floor.  First impression when I opened the box was - this is all cheap materials.  I thought to myself, I got what I paid for.  But I decided to give it a try anyway.  The vacuum was so weak that it barely can pick up all the trash/dust.  Could there be something defective on my unit?  Doesn't seem like it.  Motor and everything was running fine.  I am returning this vacuum and would not recommend this to anyone.  I don't know how it got over 4+ stars from nearly 30 people.  I usually can rely on Amazon reviews but not on this vacuum.  What a disappointment and big waste of time."

In [32]:
text_question_2 = high_quality["reviewText"][1123475]
text_question_2

'First time I\'ve tired these and the cloth is stuck in the line. Does anyone know how to best unclog the "cleaning" cloth from the line?\n\nEdit: I googled my issue and ended up hooking my regular vacuum up to the central vac outlet to try sucking the cloth back out. The cloth didn\'t come back up, but some other gunk did. I then plugged the central vac hose back in and heard material unclog and end up in the central vac unit. It looks like it was so clogged that the cloth could\'t pass though until I suctioned it the other direction to get it all moving. Now it is working well and I will give the cloths another try.'

In [33]:
from transformers import pipeline

qa_pipeline = pipeline("question-answering", model="distilbert-base-uncased-distilled-squad", framework="pt")

Device set to use mps:0


In [34]:
question = "How can I unclog the cleaning cloth from the line?"

context = text_question_2

response = qa_pipeline(question=question, context=context)

print("Question:", question)
print("Answer:", response['answer'])

Question: How can I unclog the cleaning cloth from the line?
Answer: plugged the central vac hose


In [35]:
question = "Could there be something defective on my unit?"

context = text_question

response = qa_pipeline(question=question, context=context)

print("Question:", question)
print("Answer:", response['answer'])

Question: Could there be something defective on my unit?
Answer: Could there be something defective on my unit?  Doesn't seem like it


In [36]:
question = "Is this a good vacuum cleaner?"

context = text_question

response = qa_pipeline(question=question, context=context)

print("Question:", question)
print("Answer:", response['answer'])

Question: Is this a good vacuum cleaner?
Answer: I usually can rely on Amazon reviews but not on this vacuum
